# The Bloch sphere
**Quantum Computing · Week 2 (Gates and the Bloch sphere) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qc/w2d1.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Every single-qubit state can be written with two angles:
$|\psi\rangle = \cos\frac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\frac{\theta}{2}\,|1\rangle$.
$\theta$ is the latitude (from the north pole) and $\varphi$ the longitude on a globe called the Bloch sphere.

In [ ]:
import numpy as np
from qiskit.quantum_info import Statevector

def bloch_state(theta, phi):
    """The qubit state at latitude theta and longitude phi on the Bloch sphere."""
    return Statevector([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])

famous = {"|0>": (0, 0), "|1>": (np.pi, 0), "|+>": (np.pi / 2, 0),
          "|->": (np.pi / 2, np.pi), "|+i>": (np.pi / 2, np.pi / 2)}
for name, (theta, phi) in famous.items():
    sv = bloch_state(theta, phi)
    print(f"{name:>5}: amplitudes {np.round(sv.data, 3)}   P(0) = {sv.probabilities()[0]:.2f}")

Qiskit can draw the sphere for you:

In [ ]:
from qiskit.visualization import plot_bloch_multivector

plot_bloch_multivector(bloch_state(np.pi / 3, 2 * np.pi / 3), title="theta = 60°, phi = 120°")

## Phase is invisible to a single measurement...
$|+\rangle$ and $|-\rangle$ sit on opposite sides of the equator, yet they give exactly
the same 0/1 statistics.

In [ ]:
plus = Statevector.from_label("+")
minus = Statevector.from_label("-")
print("|+> probabilities:", plus.probabilities().round(3))
print("|-> probabilities:", minus.probabilities().round(3))
print("overlap |<+|->|^2 =", round(abs(plus.inner(minus)) ** 2, 3), " (0 means: completely different states!)")

## ...and a *global* phase changes nothing at all
Multiplying the whole state by $e^{i\alpha}$ gives the same physical state.
`equiv` checks equality up to such a global phase.

In [ ]:
psi = bloch_state(1.0, 0.5)
psi_shifted = Statevector(np.exp(1j * 0.9) * psi.data)
print("same amplitudes?", np.allclose(psi.data, psi_shifted.data))
print("same physical state?", psi.equiv(psi_shifted))

## Your turn: from amplitudes to the arrow
The arrow's coordinates come straight from the amplitudes $\alpha, \beta$:
$x = 2\,\mathrm{Re}(\bar\alpha\beta)$, $y = 2\,\mathrm{Im}(\bar\alpha\beta)$, $z = |\alpha|^2 - |\beta|^2$.
Write `bloch_vector` and test it on four states.

In [ ]:
def bloch_vector(state):
    alpha, beta = state.data
    x = y = z = 0.0          # TODO: the three formulas above (np.conj, .real, .imag, abs)
    return np.round([x, y, z], 3)

for name, (theta, phi) in {"|0>": (0, 0), "|+>": (np.pi / 2, 0), "|+i>": (np.pi / 2, np.pi / 2),
                           "halfway |0>-|+i>": (np.pi / 4, np.pi / 2)}.items():
    print(f"{name:>17}: {bloch_vector(bloch_state(theta, phi))}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def bloch_vector(state):
    alpha, beta = state.data
    c = np.conj(alpha) * beta
    x, y, z = 2 * c.real, 2 * c.imag, abs(alpha) ** 2 - abs(beta) ** 2
    return np.round([x, y, z], 3)

for name, (theta, phi) in {"|0>": (0, 0), "|+>": (np.pi / 2, 0), "|+i>": (np.pi / 2, np.pi / 2),
                           "halfway |0>-|+i>": (np.pi / 4, np.pi / 2)}.items():
    print(f"{name:>17}: {bloch_vector(bloch_state(theta, phi))}")
print("Check: the arrow is (sin θ cos φ, sin θ sin φ, cos θ). For θ = 45°, φ = 90° that's (0, 0.707, 0.707).")
print("And z = P(0) - P(1): the height of the arrow is the difference of the two probabilities.")